# AI-3 Session 1.1 — Query Refinement & Chunk Enrichment

Welcome to AI-3! In AI-2, you built a RAG pipeline. It works. Today we make it work **better**.

**What we're building today:**
- `hyde.py` — HyDE (Hypothetical Document Embeddings) for query-time improvement
- `enriched.py` — Question enrichment for ingestion-time improvement

Let's start by making sure your environment is ready.

In [1]:
# Path setup — notebook is in notebooks/1_1/, pipeline is at repo root
import sys, os
from pathlib import Path

repo_root = str(Path().resolve().parent.parent)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.chdir(repo_root)  # So .env and chroma_db paths resolve

from dotenv import load_dotenv
load_dotenv()
print(f"Working from: {os.getcwd()}")

Working from: /Users/thayes-mac/Documents/GitHub/ai3-fullstack


In [2]:
# Verify your environment
from pipeline.generation.generate import call_claude
from pipeline.retrieval.naive import naive_retrieve
from pipeline.embeddings.embed import embed_texts
from pipeline.ingestion.store import get_collection

print("All imports OK!")

/Users/thayes-mac/Documents/GitHub/ai3-fullstack/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK!


In [3]:
# Verify ChromaDB is loaded
collection = get_collection()
print(f"ChromaDB: {collection.count()} chunks loaded")

ChromaDB: 173 chunks loaded


In [4]:
# Verify API keys work (makes a real API call)
response = call_claude("Say hello in exactly 3 words.")
print(f"Claude says: {response}")

Claude says: Hello to you.


In [5]:
# Verify Voyage AI connectivity
test_embedding = embed_texts(["test connectivity"])
print(f"Voyage AI: OK (embedding dim={len(test_embedding[0])})")

Voyage AI: OK (embedding dim=512)


In [7]:
# Verify Phoenix configuration
import os
from dotenv import load_dotenv
load_dotenv()

phoenix_key = os.getenv("PHOENIX_API_KEY")
phoenix_endpoint = os.getenv("PHOENIX_COLLECTOR_ENDPOINT")
phoenix_project = os.getenv("PHOENIX_PROJECT_NAME")

checks = []
if phoenix_key:
    checks.append("API Key: Set")
else:
    checks.append("API Key: MISSING -- go to app.phoenix.arize.com > Settings > API Keys")

if phoenix_endpoint:
    checks.append(f"Endpoint: {phoenix_endpoint}")
else:
    checks.append("Endpoint: MISSING")

if phoenix_project:
    checks.append(f"Project: {phoenix_project}")
else:
    checks.append("Project: MISSING -- set PHOENIX_PROJECT_NAME=ai3-yourname")

for c in checks:
    print(f"  Phoenix {c}")

if all([phoenix_key, phoenix_endpoint, phoenix_project]):
    print("\nPhoenix: All configured!")
else:
    print("\nPhoenix: Fix the MISSING items above in your .env file")

  Phoenix API Key: Set
  Phoenix Endpoint: https://app.phoenix.arize.com/s/tyler-hayes
  Phoenix Project: ai3-tylerhayes

Phoenix: All configured!


### Setup Complete

All five checks should pass:
1. Pipeline imports
2. ChromaDB loaded (173 chunks)
3. Claude API responding
4. Voyage AI returning embeddings
5. Phoenix configured

If anything is missing, fix it now before we continue.

## The Embedding Gap

Remember the PCA demo from AI-2 Session 3.2? Questions and answers cluster in **different neighborhoods** in embedding space.

- Questions encode interrogative structure: "What...", "How...", "When..."
- Answers encode declarative structure: "Employees receive...", "The policy states..."
- Same topic, **different embedding neighborhoods**

Let's see this in action with your current (naive) retrieval.

In [8]:
# Run naive retrieval — observe what comes back
query = "What is the vacation policy?"
results = naive_retrieve(query, top_k=5)

print(f"Query: {query}\n")
for r in results:
    source = r["metadata"].get("source", "Unknown")
    score = r["score"]
    text_preview = r["text"][:120]
    print(f"  [{score:.3f}] {source}")
    print(f"           {text_preview}...\n")

Query: What is the vacation policy?

  [0.482] vacation_policy_2023.md
           benefits based on their scheduled hours relative to a standard 40-hour week.

## Paid Time Off Allowances

### Vacation ...

  [0.473] vacation_policy_2025.md
           Holiday must be scheduled at least one week in advance and approved by the employee's manager.

## Bereavement and Jury ...

  [0.471] vacation_policy_2023.md
           the previous PTO policy (HR-2021-004) effective January 1, 2023.

## Eligibility All full-time employees who have comple...

  [0.463] vacation_policy_2025.md
           this policy. Part-time employees working 20 or more hours per week receive prorated benefits based on their scheduled ho...

  [0.463] vacation_policy_2023.md
           company holiday. When a holiday falls on a weekend, the nearest weekday will be observed.

## Bereavement and Jury Duty ...



In [9]:
# Try a query where the gap is larger
query2 = "How does the company handle remote work requests?"
results2 = naive_retrieve(query2, top_k=5)

print(f"Query: {query2}\n")
for r in results2:
    source = r["metadata"].get("source", "Unknown")
    score = r["score"]
    text_preview = r["text"][:120]
    print(f"  [{score:.3f}] {source}")
    print(f"           {text_preview}...\n")

Query: How does the company handle remote work requests?

  [0.464] remote_work_policy.md
           during peak periods, client engagements, or team events. Such requirements should be communicated at least one week in a...

  [0.456] remote_work_policy.md
           remote work. This policy outlines expectations, eligibility, and support for employees participating in the hybrid work ...

  [0.431] remote_work_policy.md
           onboarding and team integration. Contractors and temporary employees follow the terms of their individual agreements.

#...

  [0.422] remote_work_policy.md
           -issued laptops. Employees using personal devices must install the approved VPN client and follow the configuration guid...

  [0.411] remote_work_policy.md
           # Northbrook Partners — Remote and Hybrid Work Policy

**Policy Number:** OPS-2024-007
**Effective Date:** June 1, 2024
...



### What did you notice?

The results include chunks that *mention* the topic but aren't necessarily the **direct answers**. The query embedding lands near other question-like text, not near the answer text.

**Today's goal:** Close this gap with two techniques.
- **HyDE** — transform the query to sound like an answer (query time)
- **Question Enrichment** — transform the index to include questions (ingestion time)

## Solution 1: HyDE (Hypothetical Document Embeddings)

**The idea:** Before searching, ask Claude "What would a good answer to this question look like?" Then embed that hypothetical answer — not the original question — and use it for retrieval.

The hypothetical answer:
- Does NOT need to be correct
- Needs to SOUND like the kind of document that contains the real answer
- Lands in "answer space" near the real answers

```
NAIVE:    Question -> [embed] -> search (lands in question space)
HyDE:     Question -> [Claude] -> hypothetical answer -> [embed] -> search (lands in answer space)
```

### Let's build `generate_hypothetical_answer()`

This function takes a question and asks Claude to generate a hypothetical answer — a passage that sounds like it came from a real document.

In [ ]:
import anthropic

client = anthropic.Anthropic()


def generate_hypothetical_answer(question: str, domain: str = "company") -> str:
    """Generate a hypothetical answer for embedding-based retrieval.

    The answer does NOT need to be correct. It needs to SOUND like
    the kind of document that contains the real answer.
    """
    response = ...
    pass

In [11]:
# Test it — what does the hypothetical answer look like?
test_question = "What is the vacation policy?"
hypothetical = generate_hypothetical_answer(test_question)
print(f"Question: {test_question}")
print(f"\nHypothetical answer:\n  {hypothetical}")

Question: What is the vacation policy?

Hypothetical answer:
  **Vacation Policy**

Full-time employees accrue paid time off (PTO) at a rate of 15 days per year during their first three years of employment, increasing to 20 days per year after three years of service and 25 days per year after seven years of service. PTO accrues bi-weekly and can be used for vacation, personal days, or sick leave, with a maximum rollover of 5 unused days into the following calendar year. Employees must submit vacation requests through the HR portal at least two weeks in advance for approval by their direct manager.


### Now let's build `hyde_retrieve()`

This function:
1. Generates a hypothetical answer
2. Embeds the hypothetical answer (NOT the question!)
3. Searches ChromaDB with that embedding
4. Returns results

In [ ]:
def hyde_retrieve(question: str, n_results: int = 5, domain: str = "company") -> list[dict]:
    """Retrieve using HyDE: embed a hypothetical answer instead of the question."""
    # Step 1: Generate hypothetical answer

    # Step 2: Embed the hypothetical answer (NOT the question!)

    # Step 3: Search ChromaDB with the hypothetical answer embedding

    # Step 4: Format results
    pass

In [ ]:
# Compare naive vs HyDE
query = "Who is the CEO and what are their priorities?"

# Naive
naive_results = naive_retrieve(query, top_k=3)
print("=== NAIVE TOP 3 ===")
for r in naive_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')}")
    print(f"           {r['text'][:100]}...\n")

# HyDE
hyde_results = hyde_retrieve(query, n_results=3)
print("=== HYPOTHETICAL ANSWER ===")
print(f"  {hyde_results[0]['hyde_answer']}\n")
print("=== HYDE TOP 3 ===")
for r in hyde_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')}")
    print(f"           {r['text'][:100]}...\n")

=== NAIVE TOP 3 ===
  [0.400] memo_ceo_annual_kickoff.md
           this team has met or exceeded its goals. I have no doubt we will do it again.

## What This Means fo...

  [0.380] board_meeting_q4_2024.md
           position. Detailed plans for each priority will be shared with the board at the next quarterly meeti...

  [0.375] memo_ceo_annual_kickoff.md
           the talent, the clients, the financial position, and now the strategic clarity to make 2025 our best...

=== HYPOTHETICAL ANSWER ===
  **Executive Leadership and Strategic Priorities**

David Chen currently serves as Chief Executive Officer of the company, a position he has held since January 2021. His primary strategic priorities include accelerating digital transformation initiatives across all business units, expanding our market presence in the Asia-Pacific region through strategic partnerships and acquisitions, and driving operational excellence while maintaining our commitment to sustainable and environmentally res

In [ ]:
# Another example 
query = "What are the performance review procedures?"

naive_results = naive_retrieve(query, top_k=3)
hyde_results = hyde_retrieve(query, n_results=3)

print(f"Query: {query}\n")
print("=== NAIVE TOP 3 ===")
for r in naive_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')} — {r['text'][:80]}...")
print()
print("=== HYDE TOP 3 ===")
for r in hyde_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')} — {r['text'][:80]}...")
print()
print("Notice: Naive's 3rd result is from expense_policy.md — completely irrelevant!")
print("HyDE keeps all results in employee_handbook.md where procedures actually live.")

Query: What are the performance review procedures?

=== NAIVE TOP 3 ===
  [0.527] employee_handbook.md — using a standardized rubric that covers: quality of work, collaboration, client ...
  [0.464] employee_handbook.md — :00 PM. This is enforced via automatic calendar blocks provisioned by IT. Client...
  [0.298] expense_policy.md — ulent expense claims or repeated policy violations will result in disciplinary a...

=== HYDE TOP 3 ===
  [0.574] employee_handbook.md — :00 PM. This is enforced via automatic calendar blocks provisioned by IT. Client...
  [0.550] employee_handbook.md — using a standardized rubric that covers: quality of work, collaboration, client ...
  [0.407] vacation_policy_2023.md — day. For absences of three or more consecutive days, a physician's note may be r...

Notice: Naive's 3rd result is from expense_policy.md — completely irrelevant!
HyDE keeps all results in employee_handbook.md where procedures actually live.


### When does HyDE help? When doesn't it?

Try a **vague** query — HyDE amplifies whatever signal is in the question. No signal = no improvement.

Think about:
- Which queries show the biggest improvement?
- Which queries show no difference?
- What does the hypothetical answer look like for vague vs specific questions?

## Solution 2: Question Enrichment at Ingestion Time

**The idea:** Instead of transforming the query, transform the **index**. At ingestion time, ask Claude "What questions does this chunk answer?" Store those questions alongside the chunk.

At query time, the user's question matches against the *generated questions* — questions match questions, same neighborhood.

```
INGESTION:  Chunk -> [Claude: "what questions?"] -> questions
            -> [Embed chunk + questions] -> Store

QUERY TIME: Question -> [Embed] -> matches generated questions (free!)
```

Key difference from HyDE: **zero cost at query time**. You pay upfront at ingestion.

### Let's build `generate_questions_for_chunk()`

This function takes a chunk of text and asks Claude: "What questions does this text answer?"

In [ ]:
ENRICHED_COLLECTION = "northbrook_enriched"


def generate_questions_for_chunk(chunk_text: str, n_questions: int = 3) -> list[str]:
    """Generate questions that this chunk answers.

    Args:
        chunk_text: The text of the chunk
        n_questions: Number of questions to generate

    Returns:
        List of question strings
    """
    # Call Claude asking what questions this text answers.
    # Parse the response into a list of strings.
    pass

In [ ]:
# Get a sample chunk from ChromaDB and generate questions for it
sample = get_collection().get(limit=1, include=["documents", "metadatas"])
sample_text = sample["documents"][0]
sample_source = sample["metadatas"][0].get("source", "Unknown")

print(f"Source: {sample_source}")
print(f"Chunk: {sample_text[:200]}...\n")

questions = generate_questions_for_chunk(sample_text)
print("Generated questions:")
for q in questions:
    print(f"  - {q}")

### Now let's build `enrich_and_store()`

This function processes chunks, generates questions for each, creates combined embeddings, and stores them in a **new** ChromaDB collection (`northbrook_enriched`).

This is an **ingestion-time** operation. Run it once, and every query benefits.

In [ ]:
# Get chunks from the existing collection for enrichment
original = get_collection()
data = original.get(include=["documents", "metadatas"], limit=20)
chunks = [
    {"text": doc, "metadata": meta}
    for doc, meta in zip(data["documents"], data["metadatas"])
]
print(f"Got {len(chunks)} chunks for enrichment")

In [ ]:
import chromadb
from pathlib import Path

_REPO_ROOT = Path().resolve()
CHROMA_PATH = str(_REPO_ROOT / "chroma_db")


def enrich_and_store(chunks: list[dict], collection_name: str = ENRICHED_COLLECTION) -> None:
    """Process chunks with enrichment and store in ChromaDB.

    For each chunk:
      1. Generate questions the chunk answers
      2. Create combined text: chunk + questions
      3. Embed the combined text
      4. Store in a NEW collection with enrichment metadata

    Args:
        chunks: List of dicts with keys: text, metadata
        collection_name: Name for the enriched collection
    """
    # Get or create the enriched collection
    # Loop through chunks:
    #   - Generate questions
    #   - Combine: chunk text + "\n\nQuestions this answers:\n" + questions
    #   - Embed the COMBINED text
    #   - Store with metadata including generated_questions
    pass

In [ ]:
# Enrich 20 chunks (takes ~30-60 seconds)
enrich_and_store(chunks)

### Finally, `enriched_retrieve()`

This function queries the enriched collection. It's structurally identical to naive retrieval — but the collection has enriched embeddings that include question context.

In [ ]:
def enriched_retrieve(question: str, n_results: int = 5, collection_name: str = ENRICHED_COLLECTION) -> list[dict]:
    """Retrieve from the enriched collection.

    Just embed the question (the enrichment is in the index) and search.

    Args:
        question: The user's question
        n_results: Number of chunks to retrieve
        collection_name: The enriched collection name

    Returns:
        List of dicts with keys: text, metadata, score
    """
    # Embed the question
    # Query the enriched collection
    # Format and return results
    pass

In [ ]:
# Three-way comparison — enrichment wins here, HyDE actually fails!
query = "What happened at the Q4 board meeting?"

print(f"Query: {query}\n")

# Naive
naive_results = naive_retrieve(query, top_k=3)
print("=== NAIVE ===")
for r in naive_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')} — {r['text'][:80]}...")

# HyDE
hyde_results = hyde_retrieve(query, n_results=3)
print("\n=== HYDE ===")
print(f"  Hypothetical: {hyde_results[0]['hyde_answer'][:100]}...")
for r in hyde_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')} — {r['text'][:80]}...")

# Enriched
enriched_results = enriched_retrieve(query, n_results=3)
print("\n=== ENRICHED ===")
for r in enriched_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')} — {r['text'][:80]}...")

print("\nNotice: HyDE may MISS the Q4 board meeting doc (hypothetical hallucinated).")
print("Enrichment retrieves the right docs because generated questions match directly.")

## Save to Python Modules

Your notebook code works. Now let's make it a proper module that your app can import.

### `pipeline/retrieval/hyde.py`

Copy your working functions into this file. The file already exists with imports set up — add your function implementations.

```python
"""
hyde.py -- Hypothetical Document Embeddings

Transform queries by generating hypothetical answers, then embed the
hypothetical answer for retrieval instead of the raw question.
"""

import anthropic
from pipeline.embeddings.embed import embed_texts
from pipeline.ingestion.store import get_collection

client = anthropic.Anthropic()


def generate_hypothetical_answer(question: str, domain: str = "company") -> str:
    # Your implementation from the notebook
    ...

def hyde_retrieve(question: str, n_results: int = 5, domain: str = "company") -> list[dict]:
    # Your implementation from the notebook
    ...
```

### `pipeline/retrieval/enriched.py`

```python
"""
enriched.py -- Question Enrichment at Ingestion Time

At ingestion time, generate questions each chunk answers.
Embed those questions alongside the chunk for better query matching.
"""

import anthropic
from pipeline.embeddings.embed import embed_texts
from pipeline.ingestion.store import get_collection

client = anthropic.Anthropic()

ENRICHED_COLLECTION = "northbrook_enriched"


def generate_questions_for_chunk(chunk_text: str, n_questions: int = 3) -> list[str]:
    # Your implementation from the notebook
    ...

def enrich_and_store(chunks: list[dict], collection_name: str = ENRICHED_COLLECTION) -> None:
    # Your implementation from the notebook
    ...

def enriched_retrieve(question: str, n_results: int = 5, collection_name: str = ENRICHED_COLLECTION) -> list[dict]:
    # Your implementation from the notebook
    ...
```

In [ ]:
# After saving your .py files, verify they import correctly
# (restart kernel first if needed)
from pipeline.retrieval.hyde import hyde_retrieve
from pipeline.retrieval.enriched import enriched_retrieve

print("Module imports OK!")

## What You Built Today

- **`hyde.py`** — transforms queries into answer space (online, per-query cost)
- **`enriched.py`** — transforms index into question space (offline, per-chunk cost)
- Both close the embedding gap you saw in AI-2's PCA demo
- Both improve retrieval over the naive baseline

## Next Session — 1.2: Proving Improvement

You improved retrieval. But "I can see it looks better" is not engineering. Next session you build the evidence:

- **Golden test sets** — curated question/answer pairs
- **Before/after evaluation** — quantify the improvement
- **Regression framework** — catch when changes break things

**Questions to think about:**
1. How do you **prove** retrieval is better, not just feel better?
2. If you change the pipeline tomorrow, how do you know you didn't break what was working?
3. Can you measure the embedding gap closing?

**Make sure both `hyde.py` and `enriched.py` are working before Session 1.2.**